<a href="https://colab.research.google.com/github/nmit-1NT23CS128/Social-Media-Comment-Moderation/blob/main/Social_Media_Comment_Moderation_BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install sympy==1.14.0 --force-reinstall
from transformers import DistilBertForSequenceClassification, Trainer, TrainingArguments

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 49.9 MB/s eta 0:00:00
  Attempting uninstall: mpmath
    Found existing installation: mpmath 1.3.0
    Uninstalling mpmath-1.3.0:
      Successfully uninstalled mpmath-1.3.0
  Attempting uninstall: sympy
    Found existing installation: sympy 1.14.0
    Uninstalling sympy-1.14.0:
      Successfully uninstalled sympy-1.14.0


In [2]:
!pip install transformers datasets torch scikit-learn pandas numpy


In [ ]:
pip install toxic-comment-collection

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
#from huggingface_hub import notebook_login
#notebook_login()


In [ ]:
import pandas as pd

data_path = "/content/drive/MyDrive/train.csv"
df = pd.read_csv(data_path)

df.head()



In [ ]:
from toxic_comment_collection import get_dataset

# Download the basile2019 dataset
print("Downloading 'basile2019' dataset...")
get_dataset('basile2019')
print("'basile2019' dataset downloaded.")

# Load the downloaded dataset
df_new = pd.read_csv("./files/basile2019/basile2019en.csv", sep="\t")

# Process labels for the new dataset
# The 'labels' column contains lists like ['hate'] or []. Convert to binary (1 if 'hate' is present, 0 otherwise)
df_new['label'] = df_new['labels'].apply(lambda x: 1 if "'hate'" in x else 0)

# Rename 'text' column to 'comment_text' to match the original DataFrame
df_new = df_new.rename(columns={'text': 'comment_text'})

# Select only the relevant columns for concatenation
df_new = df_new[['comment_text', 'label']]

# Concatenate the new dataset with the original DataFrame
df = pd.concat([df, df_new], ignore_index=True)

print("Combined DataFrame head:")
display(df.head())
print("Combined DataFrame tail:")
display(df.tail())

In [ ]:
from toxic_comment_collection import get_dataset

# Download the 'albadi2018' dataset
print("Downloading 'albadi2018' dataset...")
get_dataset('albadi2018')
print("'albadi2018' dataset downloaded.")

# Load the downloaded dataset (using the 'train' split as an example)
df_albadi = pd.read_csv("./files/albadi2018/albadi2018ar_train.csv", sep="\t")

# Process labels for the new dataset
# Assuming 'labels' column and 'hate' detection similar to basile2019, adjust if different
# If the 'labels' column does not exist or has a different format, this will need adjustment after running.
if 'labels' in df_albadi.columns:
    df_albadi['label'] = df_albadi['labels'].apply(lambda x: 1 if "'hate'" in str(x) else 0)
elif 'offensive' in df_albadi.columns: # Sometimes datasets have direct toxicity labels
    df_albadi['label'] = df_albadi['offensive'] # Assuming 1 for offensive, 0 otherwise
else:
    # Fallback if specific label column not found, might need manual inspection
    print("Warning: 'labels' or 'offensive' column not found in albadi2018. Defaulting to 0 for all labels.")
    df_albadi['label'] = 0

# Rename 'text' column to 'comment_text' to match the original DataFrame
df_albadi = df_albadi.rename(columns={'text': 'comment_text'})

# Select only the relevant columns for concatenation
df_albadi = df_albadi[['comment_text', 'label']]

# Concatenate the new dataset with the existing DataFrame
df = pd.concat([df, df_albadi], ignore_index=True)

print("Combined DataFrame after adding albadi2018 head:")
display(df.head())
print("Combined DataFrame after adding albadi2018 tail:")
display(df.tail())

ModuleNotFoundError: No module named 'toxic_comment_collection'

In [ ]:
toxicity_columns = [
    'toxic',
    'severe_toxic',
    'obscene',
    'threat',
    'insult',
    'identity_hate'
]

df['label'] = df[toxicity_columns].max(axis=1)


In [ ]:
texts = df['comment_text'].fillna("")
labels = df['label']


In [ ]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42
)


In [ ]:
print(train_labels.value_counts())


In [ ]:
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')


In [ ]:
import torch


In [ ]:
class ToxicDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts = texts.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True)
        self.tokenizer = tokenizer

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding='max_length',
            max_length=128,
            return_tensors='pt'
        )

        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item['labels'] = torch.tensor(self.labels[idx])

        return item

    def __len__(self):
        return len(self.labels)


In [ ]:
train_dataset = ToxicDataset(train_texts, train_labels, tokenizer)
test_dataset = ToxicDataset(test_texts, test_labels, tokenizer)

In [ ]:
print(len(train_dataset))
print(len(test_dataset))


In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)


In [ ]:
training_args = TrainingArguments(
    output_dir='./results',

    num_train_epochs=1,

    per_device_train_batch_size=4,   # reduced
    per_device_eval_batch_size=4,    # reduced

    gradient_accumulation_steps=4,   # acts like batch size 16

    learning_rate=2e-5,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=100,

    fp16=True   # ⭐ THIS IS STEP 5 (add this line)
)



In [ ]:
from transformers import Trainer


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)


In [ ]:
trainer.train()

In [ ]:
model.save_pretrained("/content/drive/MyDrive/toxic_model")
tokenizer.save_pretrained("/content/drive/MyDrive/toxic_model")

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

predictions = trainer.predict(test_dataset)
preds = predictions.predictions.argmax(-1)

accuracy = accuracy_score(test_labels, preds)
precision, recall, f1, _ = precision_recall_fscore_support(test_labels, preds, average='binary')

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

In [ ]:
# training_args = TrainingArguments(
#     output_dir='./results',
#     num_train_epochs=3,          # increase epochs
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=16,
#     learning_rate=2e-5,          # KEY CHANGE
#     eval_strategy="epoch",
#     save_strategy="epoch",
#     weight_decay=0.01            # regularization
# )


In [ ]:
# from transformers import DistilBertForSequenceClassification, DistilBertTokenizer

# # Re-initialize tokenizer if it's not defined (due to potential kernel restart)
# tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# # Re-initialize model if it's not defined (due to potential kernel restart)
# model = DistilBertForSequenceClassification.from_pretrained(
#     'distilbert-base-uncased',
#     num_labels=2
# )
# model.save_pretrained("/content/drive/MyDrive/toxic_model")
# tokenizer.save_pretrained("/content/drive/MyDrive/toxic_model")

In [ ]:
!mkdir BERT_Toxic_Project

In [ ]:
!ls

In [ ]:
%cd BERT_Toxic_Project

In [ ]:
pip install matplotlib scikit-learn

In [ ]:
%%writefile app.py
import streamlit as st
import numpy as np
import torch
import torch.nn.functional as F
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix



# Page config + styling

st.set_page_config(page_title="Toxic Comment Detector", layout="centered")

st.markdown("""

<style>
.main {
    background-color: #0e1117;
}
.stTextArea textarea {
    font-size: 16px;
}
</style>

""", unsafe_allow_html=True)

# Load model

model = DistilBertForSequenceClassification.from_pretrained(
"/content/drive/MyDrive/toxic_model"
)

tokenizer = DistilBertTokenizer.from_pretrained(
"/content/drive/MyDrive/toxic_model"
)

model.eval()

# Sidebar navigation

page = st.sidebar.selectbox(
"Select Page",
["Toxic Comment Detector", "Model Performance"]
)

# -------------------------------

# PAGE 1: LIVE PREDICTION

# -------------------------------

if page == "Toxic Comment Detector":
    st.markdown("## 🛡 AI Toxic Comment Moderator")
    st.caption("Detects and filters harmful comments in real-time")

    def normalize_text(text):
        text = text.lower()
        text = text.replace("ur", "you are")
        text = text.replace("u ", "you ")
        return text


    user_input = st.text_area("Enter Comment", key="input_text")

    if user_input.strip() == "":
        st.info("Please enter a comment")
    else:
        if st.button("Predict"):

            processed = normalize_text(user_input)



            inputs = tokenizer(
                processed,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=128
            )

            # Spinner added here
            with st.spinner("Analyzing comment..."):
                with torch.no_grad():
                    outputs = model(**inputs)
                    probs = F.softmax(outputs.logits, dim=1)
                    toxic_prob = probs[0][1].item()

            if toxic_prob >0.85:
              st.error("Highly Toxic -🚨 Comment Blocked")
              decision = "Automatically Hidden"

            elif toxic_prob >0.4:
                st.warning("Moderately Toxic - ⚠ Comment Flagged")
                decision = "Blocked (Pending Risk)"

            else:
                st.success("Less Toxic - ✅ Comment Approved")
                decision = "Published"

            # Display probability
            st.write("Toxic Probability:", round(toxic_prob, 3))
            st.progress(int(toxic_prob * 100))

            st.caption(f"Confidence Score: {round(toxic_prob, 3)}")
            st.progress(int(toxic_prob * 100))

            # System Action
            st.write(f"🛠 System Action: **{decision}**")

# -------------------------------

# PAGE 2: MODEL DASHBOARD

# -------------------------------

else:

    st.title("📊 Model Performance Dashboard")

    accuracy = 0.951
    precision = 0.763
    recall = 0.754
    f1 = 0.758

    st.metric("Accuracy", accuracy)
    st.metric("Precision", precision)
    st.metric("Recall", recall)
    st.metric("F1 Score", f1)

    # Example confusion matrix
    y_true = [0, 0, 0, 1, 1, 1]
    y_pred = [0, 0, 1, 1, 1, 0]

    cm = confusion_matrix(y_true, y_pred)

    fig, ax = plt.subplots()
    ax.matshow(cm)

    for (i, j), val in np.ndenumerate(cm):
        ax.text(j, i, val, ha='center', va='center')

    plt.xlabel("Predicted")
    plt.ylabel("Actual")

    st.pyplot(fig)

In [ ]:
!ls

In [ ]:
model.save_pretrained("/content/drive/MyDrive/toxic_model")
tokenizer.save_pretrained("/content/drive/MyDrive/toxic_model")

In [ ]:
!pip install pyngrok

In [ ]:
!pip install pyngrok streamlit

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3BDTF8NoUigVbuJIvmmnidQzfQm_4UTY1Ysq7tMr8ihd3iAC9")

In [ ]:
!streamlit run app.py &>/content/logs.txt &

In [ ]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)
print(public_url)

In [ ]:
!ls /content/drive/MyDrive/toxic_model